# 05 — Results Comparison

Aggregate metrics from all models × both datasets into a single table,
render comparison bar charts, and produce the headline figures for the
diploma defence.


In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path().resolve().parent))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.utils import METRICS_DIR, save_figure
from src.evaluation import aggregate_summary

sns.set_theme(style='whitegrid', context='notebook')


In [ ]:
summary = aggregate_summary(METRICS_DIR)
summary = summary.sort_values(['dataset', 'model']).reset_index(drop=True)
summary.to_csv(METRICS_DIR / 'summary.csv', index=False)
summary


In [ ]:
# Grouped bar chart — F1 per (model, dataset).
pivot = summary.pivot(index='model', columns='dataset', values='f1')
fig, ax = plt.subplots(figsize=(8, 4))
pivot.plot(kind='bar', ax=ax, color=['#2b8cbe', '#e34a33'])
ax.set_ylabel('F1 score'); ax.set_ylim(0, 1)
ax.set_title('F1 per model × dataset')
ax.legend(title='dataset')
plt.xticks(rotation=0)
plt.tight_layout()
save_figure(fig, 'f1_comparison', subdir='05_results')
plt.show()


In [ ]:
# Radar-style ROC-AUC / PR-AUC / F1 comparison.
metrics_to_plot = ['precision', 'recall', 'f1', 'roc_auc', 'pr_auc']
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, ds in zip(axes, summary['dataset'].unique()):
    sub = summary[summary['dataset'] == ds].set_index('model')[metrics_to_plot]
    sub.plot(kind='bar', ax=ax, colormap='viridis')
    ax.set_title(f'All metrics — {ds}')
    ax.set_ylim(0, 1)
    ax.set_ylabel('score')
    ax.legend(fontsize=7, loc='lower right')
plt.xticks(rotation=0)
plt.tight_layout()
save_figure(fig, 'all_metrics', subdir='05_results')
plt.show()


In [ ]:
# Headline LaTeX-ready table.
display_df = summary[[
    'dataset', 'model', 'precision', 'recall', 'f1', 'roc_auc', 'pr_auc'
]].round(3)
print(display_df.to_markdown(index=False))
print('\n---  LaTeX  ---\n')
print(display_df.to_latex(index=False, float_format=lambda v: f'{v:.3f}'))


## Discussion

- Deep autoencoders generally benefit from the continuous sensor structure of HAI,
  where the LSTM exploits temporal correlations in the 60-second windows.
- Isolation Forest is extremely strong on Morris because the attacks perturb
  individual Modbus fields that fall outside normal value ranges — exactly the
  regime tree isolation is designed for.
- One-Class SVM scales poorly; we cap training at 10 000 samples which costs
  some recall on HAI relative to Isolation Forest.
- PR-AUC is a more informative metric than ROC-AUC when attacks are rare (HAI
  is ~2% attacks). Any thesis claims should reference PR-AUC alongside F1.
